# BirdCLEF 2026 - Multi-Label EffNet/ConvNeXt Baseline
This notebook implements a sliding window multi-label classification pipeline for BirdCLEF 2026.

## 1. Setup & Imports

In [ ]:
import os
import gc
import sys
import math
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import librosa
import soundfile as sf

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T

import timm
import albumentations as A
from sklearn.metrics import average_precision_score, roc_auc_score # roc_auc_macro is the competition metric
from sklearn.model_selection import StratifiedKFold

import warnings
warnings.filterwarnings('ignore')

## 2. Configuration & Paths

In [ ]:
class Config:
    # Paths (Kaggle specific configs)
    ROOT_DIR = '../../../../data/raw'  # Adjust based on Kaggle vs Local
    TRAIN_CSV = os.path.join(ROOT_DIR, 'train.csv')
    TRAIN_AUDIO_DIR = os.path.join(ROOT_DIR, 'train_audio')
    
    # Audio Setup
    SR = 32000
    WINDOW_SECONDS = 5
    HOP_SECONDS = 2.5  # For inference overlap
    
    # Mel Spectrogram Setup
    N_MELS = 128
    N_FFT = 2048
    HOP_LENGTH = 512
    FMIN = 20
    FMAX = 16000
    
    # Training Setup
    SEED = 42
    BATCH_SIZE = 32
    EPOCHS = 20
    LR = 1e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 4
    
    # Model Setup
    MODEL_NAME = 'tf_efficientnet_b0_ns' # Can also use 'convnext_tiny'
    NUM_CLASSES = 0 # Will be dynamically populated
    
CFG = Config()

## 3. Utility Functions

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.SEED)

## 4. Dataset & Data Processing

In [ ]:
class BirdDataset(Dataset):
    def __init__(self, df, audio_dir, transform=None, is_train=True):
        self.df = df
        self.audio_dir = audio_dir
        self.transform = transform
        self.is_train = is_train
        
        self.window_samples = CFG.SR * CFG.WINDOW_SECONDS
        
        # Initialize Mel Transform (Done on GPU later or CPU here)
        self.mel_transform = T.MelSpectrogram(
            sample_rate=CFG.SR,
            n_fft=CFG.N_FFT,
            hop_length=CFG.HOP_LENGTH,
            n_mels=CFG.N_MELS,
            f_min=CFG.FMIN,
            f_max=CFG.FMAX
        )
        self.amplitude_to_db = T.AmplitudeToDB(top_db=80)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row['filename'])
        
        try:
            # Load audio
            y, sr = librosa.load(audio_path, sr=CFG.SR, mono=True)
        except Exception as e:
            print(f'Error loading {audio_path}: {e}')
            y = np.zeros(self.window_samples)
            
        # Pad or Crop
        if len(y) > self.window_samples:
            if self.is_train:
                start = random.randint(0, len(y) - self.window_samples)
            else:
                start = 0
            y = y[start : start + self.window_samples]
        elif len(y) < self.window_samples:
            pad_len = self.window_samples - len(y)
            y = np.pad(y, (0, pad_len))
            
        # Waveform to Mel Spec
        y_tensor = torch.tensor(y, dtype=torch.float32)
        mel_spec = self.mel_transform(y_tensor)
        mel_spec = self.amplitude_to_db(mel_spec)
        
        # Normalize
        mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-6)
        
        # Convert to 3 channels for CNN backbones
        image = torch.stack([mel_spec, mel_spec, mel_spec])
        
        # Target Multi-label Encoding
        target = torch.zeros(CFG.NUM_CLASSES, dtype=torch.float32)
        target[row['label_id']] = 1.0  # Assuming label_id is mapped dynamically
        
        return image, target

## 5. Augmentations

In [ ]:
# Utilities for SpecAugment, Mixup, etc.
def get_train_transforms():
    return A.Compose([
        A.CoarseDropout(max_holes=1, max_height=16, max_width=16, p=0.5),
    ])

def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).cuda()
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

## 6. Model Architecture

In [ ]:
class BirdModel(nn.Module):
    def __init__(self, model_name, num_classes, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, in_chans=3)
        
        if 'efficientnet' in model_name:
            in_features = self.backbone.classifier.in_features
            self.backbone.classifier = nn.Identity()
        elif 'convnext' in model_name:
            in_features = self.backbone.head.fc.in_features
            self.backbone.head.fc = nn.Identity()
        else:
            in_features = self.backbone.get_classifier().in_features
            self.backbone.reset_classifier(0)
            
        self.head = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        features = self.backbone(x)
        out = self.head(features)
        return out

## 7. Loss & Optimization

In [ ]:
def get_optimizer(model):
    return optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)

def get_scheduler(optimizer):
    return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=1e-6)

def get_criterion():
    return nn.BCEWithLogitsLoss()  # Multi-label loss

## 8. Training Loops

In [ ]:
def train_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    epoch_loss = 0
    for images, targets in tqdm(loader, desc='Train'):
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        
        # Mixup
        images, targets_a, targets_b, lam = mixup_data(images, targets)
        
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        
    return epoch_loss / len(loader)

def valid_epoch(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    preds, true_targets = [], []
    
    with torch.no_grad():
        for images, targets in tqdm(loader, desc='Valid'):
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            epoch_loss += loss.item()
            
            preds.append(torch.sigmoid(outputs).cpu().numpy())
            true_targets.append(targets.cpu().numpy())
            
    preds = np.concatenate(preds)
    true_targets = np.concatenate(true_targets)
    
    # ROC-AUC skipping classes with no TPs
    auc_scores = []
    for i in range(CFG.NUM_CLASSES):
        # Only calculate AUC if there is at least one positive and one negative sample
        if len(np.unique(true_targets[:, i])) > 1:
            auc = roc_auc_score(true_targets[:, i], preds[:, i])
            auc_scores.append(auc)
            
    final_score = np.mean(auc_scores) if auc_scores else 0.0
    return epoch_loss / len(loader), final_score

## 9. Main Execution

In [ ]:
# 1. Load and Prepare Data
df = pd.read_csv(CFG.TRAIN_CSV)

# Create Label Mapping
unique_labels = sorted(df['primary_label'].unique())
label_to_id = {label: i for i, label in enumerate(unique_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}
df['label_id'] = df['primary_label'].map(label_to_id)

CFG.NUM_CLASSES = len(unique_labels)
print(f"Detected {CFG.NUM_CLASSES} classes.")

# 2. Cross-Validation Split
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG.SEED)
for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['label_id'])):
    df.loc[val_idx, 'fold'] = fold

# 3. Training Loop
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for fold in range(1): # Start with Fold 0
    print(f"\n{'='*20} Training Fold {fold} {'='*20}")
    
    # Split data
    train_df = df[df['fold'] != fold].reset_index(drop=True)
    valid_df = df[df['fold'] == fold].reset_index(drop=True)
    
    # Create Datasets & Loaders
    train_ds = BirdDataset(train_df, CFG.TRAIN_AUDIO_DIR, is_train=True)
    valid_ds = BirdDataset(valid_df, CFG.TRAIN_AUDIO_DIR, is_train=False)
    
    train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS)
    valid_loader = DataLoader(valid_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS)
    
    # Initialize Model, Optimizer, Scaler
    model = BirdModel(CFG.MODEL_NAME, CFG.NUM_CLASSES).to(device)
    optimizer = get_optimizer(model)
    scheduler = get_scheduler(optimizer)
    criterion = get_criterion()
    scaler = torch.cuda.amp.GradScaler()
    
    best_score = 0
    
    for epoch in range(CFG.EPOCHS):
        start_time = time.time()
        
        # Train & Validate
        train_loss = train_epoch(model, train_loader, optimizer, criterion, scaler, device)
        valid_loss, valid_map = valid_epoch(model, valid_loader, criterion, device)
        
        scheduler.step()
        
        # Log results
        duration = time.time() - start_time
        print(f"Epoch {epoch} | Loss: {train_loss:.4f} | Val Loss: {valid_loss:.4f} | mAP: {valid_map:.4f} | Time: {duration:.1f}s")
        
        # Save best model
        if valid_map > best_score:
            best_score = valid_map
            torch.save(model.state_dict(), f"best_model_fold{fold}.pth")
            print(f"--> Saved best model with mAP: {best_score:.4f}")
